# ABSA Transformer Training

Use this notebook on Colab or Kaggle GPU. Upload this project folder or clone it from your Git repository, then run the cells below.

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Tải toàn bộ dự án từ GitHub về môi trường Colab
!git clone https://github.com/ngocvo2511/NLP.git

Cloning into 'NLP'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 32 (delta 0), reused 32 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 1021.92 KiB | 3.28 MiB/s, done.


In [ ]:
!git pull

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 657 bytes | 657.00 KiB/s, done.
From https://github.com/ngocvo2511/NLP
   d3b6b3d..5fc7f0e  main       -> origin/main
Updating d3b6b3d..5fc7f0e
Fast-forward
 src/absa/train_transformer.py | 24 +++++++++++++++---------
 1 file changed, 15 insertions(+), 9 deletions(-)


In [ ]:
!pip install -q transformers accelerate datasets seqeval scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


Set `PROJECT_DIR` to the folder that contains `data/` and `src/`.

In [ ]:
import os
PROJECT_DIR = "/content/NLP"
os.chdir(PROJECT_DIR)
!ls

data  docs  notebooks  README.md  requirements.txt  src  tests


In [ ]:
!python -m src.absa.validate_data --data-dir data


train.jsonl
  rows: 7785
  labels: 24771
  avg labels / row: 3.18
  text length avg/max: 158.1/842
  span length avg/max: 31.5/406
  bad offsets: 81
  empty spans after clamp: 0
  rows with overlapping spans: 0
  aspects: {'GENERAL': 5637, 'PERFORMANCE': 4799, 'BATTERY': 3943, 'FEATURES': 2788, 'CAMERA': 2182, 'SER&ACC': 1875, 'DESIGN': 1497, 'PRICE': 995, 'SCREEN': 962, 'STORAGE': 93}
  polarities: {'POSITIVE': 15203, 'NEGATIVE': 7999, 'NEUTRAL': 1569}

dev.jsonl
  rows: 1112
  labels: 3582
  avg labels / row: 3.22
  text length avg/max: 156.9/850
  span length avg/max: 31.4/271
  bad offsets: 14
  empty spans after clamp: 0
  rows with overlapping spans: 0
  aspects: {'GENERAL': 802, 'PERFORMANCE': 739, 'BATTERY': 573, 'FEATURES': 383, 'CAMERA': 318, 'SER&ACC': 236, 'DESIGN': 234, 'SCREEN': 144, 'PRICE': 135, 'STORAGE': 18}
  polarities: {'POSITIVE': 2296, 'NEGATIVE': 1062, 'NEUTRAL': 224}

test.jsonl
  rows: 2225
  labels: 7043
  avg labels / row: 3.17
  text length avg/max: 159.8/

Train XLM-R first. It has a fast tokenizer with offset mappings, which makes span alignment easier.

In [ ]:
!python -m src.absa.train_transformer \
  --model-name FacebookAI/xlm-roberta-base \
  --data-dir data \
  --output-dir outputs/xlmr-absa \
  --epochs 5 \
  --batch-size 8 \
  --max-length 256

Loading weights: 100% 197/197 [00:00<00:00, 7149.90it/s]
[transformers] XLMRobertaForTokenClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
{'loss': '2.999', 'grad_norm': '10.13', 'learning_rate': '1.98e-05', 'epoch': '0.05133'}
{'loss': '2.497', 'grad_norm

In [ ]:
!python -m src.absa.predict_transformer \
  --model-dir outputs/xlmr-absa \
  --input data/dev.jsonl \
  --output outputs/xlmr_dev_predictions.jsonl

Loading weights: 100% 199/199 [00:00<00:00, 6673.22it/s]
Wrote 1112 predictions to outputs/xlmr_dev_predictions.jsonl


In [ ]:
!python -m src.absa.evaluate \
  --gold data/dev.jsonl \
  --pred outputs/xlmr_dev_predictions.jsonl \
  --json-output outputs/xlmr_dev_metrics.json

Exact span + label micro scores
  TP=2002 FP=2820 FN=1580
  Precision=0.4152 Recall=0.5589 F1=0.4764

By aspect
name                         tp     fp     fn        p        r       f1
PRICE                        42    159     93   0.2090   0.3111   0.2500
FEATURES                    188    364    195   0.3406   0.4909   0.4021
PERFORMANCE                 375    663    364   0.3613   0.5074   0.4221
SCREEN                       75    121     69   0.3827   0.5208   0.4412
SER&ACC                     129    193    107   0.4006   0.5466   0.4624
CAMERA                      196    257    122   0.4327   0.6164   0.5084
BATTERY                     341    403    232   0.4583   0.5951   0.5178
STORAGE                       8      4     10   0.6667   0.4444   0.5333
DESIGN                      144    156     90   0.4800   0.6154   0.5393
GENERAL                     504    500    298   0.5020   0.6284   0.5581

By polarity
name                         tp     fp     fn        p        r       f1

After training, download the whole `outputs/xlmr-absa` directory plus prediction and metric JSON files for reporting.